In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [2]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [3]:
df.show(10, truncate=False)

+-------+-----------+--------+-------------------+-------+-------+
|amount |category   |store   |timestamp          |tx_id  |user_id|
+-------+-----------+--------+-------------------+-------+-------+
|2382.45|żywność    |Gdańsk  |2026-05-07 08:46:22|TX46908|u015   |
|3486.07|elektronika|Gdańsk  |2026-05-07 08:10:04|TX32436|u081   |
|4334.24|elektronika|Kraków  |2026-05-07 09:32:37|TX27789|u049   |
|4173.55|książki    |Gdańsk  |2026-05-07 09:58:56|TX23960|u057   |
|1455.0 |żywność    |Warszawa|2026-05-07 08:59:22|TX74232|u083   |
|2620.2 |elektronika|Wrocław |2026-05-07 08:03:44|TX74880|u012   |
|1301.07|książki    |Kraków  |2026-05-07 08:15:24|TX62926|u053   |
|2685.53|elektronika|Kraków  |2026-05-07 08:48:16|TX64937|u078   |
|1207.65|elektronika|Warszawa|2026-05-07 10:42:09|TX89964|u003   |
|1558.8 |książki    |Warszawa|2026-05-07 08:12:13|TX46615|u082   |
+-------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [4]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
# zadanie 2.1
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2485|6335999.78|     2549.7|
|  Kraków|     2536|6388053.79|    2518.95|
|Warszawa|     2513|6325330.67|    2517.04|
| Wrocław|     2466|6250513.07|    2534.68|
+--------+---------+----------+-----------+



In [6]:
# zadanie 2.2
from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round

# Statystyki per kategoria
category_stats = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_kwot"),
        _min("amount").alias("min_kwota"),
        _max("amount").alias("max_kwota"),
        count("tx_id").alias("liczba_transakcji")
    )
    .orderBy("category")
)

category_stats.show()

+-----------+----------+---------+---------+-----------------+
|   category| suma_kwot|min_kwota|max_kwota|liczba_transakcji|
+-----------+----------+---------+---------+-----------------+
|elektronika|6461390.81|     5.81|   4995.2|             2558|
|    książki|6252432.34|     6.17|  4998.71|             2468|
|     odzież|6410645.01|     7.34|  4999.84|             2529|
|    żywność|6175429.15|     5.97|   4999.1|             2445|
+-----------+----------+---------+---------+-----------------+



In [7]:
# zadanie 3.1
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-05-07 08:00:00, 2026-05-07 09:00:00}|3322     |8417203.29|
|{2026-05-07 09:00:00, 2026-05-07 10:00:00}|3306     |8362712.65|
|{2026-05-07 10:00:00, 2026-05-07 11:00:00}|3371     |8517547.11|
|{2026-05-07 11:00:00, 2026-05-07 12:00:00}|1        |2434.26   |
+------------------------------------------+---------+----------+



In [8]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-07 08:00:00|2026-05-07 09:00:00|3322     |8417203.29|
|2026-05-07 09:00:00|2026-05-07 10:00:00|3306     |8362712.65|
|2026-05-07 10:00:00|2026-05-07 11:00:00|3371     |8517547.11|
|2026-05-07 11:00:00|2026-05-07 12:00:00|1        |2434.26   |
+-------------------+-------------------+---------+----------+



In [9]:
# zadanie 3.2 - Okna 30-minutowe per sklep
from pyspark.sql.functions import window, count, sum as _sum, round as _round, col

store_windows_30m = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN"
    )
    .orderBy("od").orderBy("store")
)

store_windows_30m.show(truncate=False)

+-------------------+-------------------+--------+---------+----------+
|od                 |do                 |store   |liczba_tx|suma_PLN  |
+-------------------+-------------------+--------+---------+----------+
|2026-05-07 09:30:00|2026-05-07 10:00:00|Gdańsk  |431      |1107895.81|
|2026-05-07 09:00:00|2026-05-07 09:30:00|Gdańsk  |397      |1015778.47|
|2026-05-07 10:00:00|2026-05-07 10:30:00|Gdańsk  |431      |1092503.17|
|2026-05-07 08:00:00|2026-05-07 08:30:00|Gdańsk  |395      |1014171.28|
|2026-05-07 08:30:00|2026-05-07 09:00:00|Gdańsk  |407      |1009006.75|
|2026-05-07 10:30:00|2026-05-07 11:00:00|Gdańsk  |424      |1096644.3 |
|2026-05-07 11:00:00|2026-05-07 11:30:00|Kraków  |1        |2434.26   |
|2026-05-07 08:30:00|2026-05-07 09:00:00|Kraków  |419      |1068244.72|
|2026-05-07 10:00:00|2026-05-07 10:30:00|Kraków  |432      |1086377.7 |
|2026-05-07 09:30:00|2026-05-07 10:00:00|Kraków  |443      |1088689.01|
|2026-05-07 09:00:00|2026-05-07 09:30:00|Kraków  |401      |1021

In [10]:
# Zadanie 3.3 - Najlepsza godzina (pod względem przychodu) dla Krakowa
from pyspark.sql.functions import desc, window, sum as _sum, round as _round, col

krakow_best_hour = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("suma_przychodu"))
    .orderBy(desc("suma_przychodu"))
)

krakow_best_hour.select(
    col("window.start").alias("godzina_start"),
    col("window.end").alias("godzina_koniec"),
    "suma_przychodu"
).show(1, truncate=False)

+-------------------+-------------------+--------------+
|godzina_start      |godzina_koniec     |suma_przychodu|
+-------------------+-------------------+--------------+
|2026-05-07 10:00:00|2026-05-07 11:00:00|2170537.34    |
+-------------------+-------------------+--------------+
only showing top 1 row



In [11]:
# zadanie 4.1
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-07 07:30:00|2026-05-07 08:30:00|1655     |4174065.54|
|2026-05-07 08:00:00|2026-05-07 09:00:00|3322     |8417203.29|
|2026-05-07 08:30:00|2026-05-07 09:30:00|3265     |8296058.15|
|2026-05-07 09:00:00|2026-05-07 10:00:00|3306     |8362712.65|
|2026-05-07 09:30:00|2026-05-07 10:30:00|3421     |8668957.5 |
|2026-05-07 10:00:00|2026-05-07 11:00:00|3371     |8517547.11|
|2026-05-07 10:30:00|2026-05-07 11:30:00|1659     |4160816.12|
|2026-05-07 11:00:00|2026-05-07 12:00:00|1        |2434.26   |
+-------------------+-------------------+---------+----------+



In [12]:
# zadanie 4.2
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)

sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)

print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# TWOJA ODPOWIEDŹ:
# Sliding window generuje więcej wierszy, ponieważ okna nakładają się na siebie (overlap). 
# W oknie tumbling nowa grupa powstaje dopiero po zakończeniu poprzedniej (co 1h). 
# W oknie sliding nowe okno tworzy się co "slideDuration" (czyli tutaj co 30 min), 
# co sprawia, że w tym samym przedziale czasu mamy dwa razy więcej punktów startowych okien. 
# Dodatkowo, ta sama transakcja jest w tym przypadku zliczana do dwóch różnych okien naraz.

Tumbling (1h):          4 okien
Sliding  (1h / 30min):  8 okien


In [13]:
# Część 5: Pytania kontrolne

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 3306 transakcji.

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") wykonuje agregację globalną – sumuje wszystkie dane dla danego sklepu 
#               z całego pliku (cały zakres czasu). 
#               groupBy(window(...), "store") wykonuje agregację czasową – dzieli dane na 
#               osobne koszyki (buckety) czasowe i liczy statystyki osobno dla każdego przedziału.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna. 
#               Transakcja z godziny 09:30:00 wpada do:
#               1. Okna [09:00:00 - 10:00:00] (09:30 jest w środku)
#               2. Okna [09:30:00 - 10:30:00] (09:30 to punkt startowy tego okna)

In [14]:
####### Praca domowa #######

# 1.
from pyspark.sql.functions import avg, col, window, round as _round

gdansk_min_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_PLN"))
    .orderBy("srednia_PLN")
)

print("Godzina z najniższą średnią w Gdańsku:")
gdansk_min_avg.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    "srednia_PLN"
).show(1, truncate=False)

Godzina z najniższą średnią w Gdańsku:
+-------------------+-------------------+-----------+
|od                 |do                 |srednia_PLN|
+-------------------+-------------------+-----------+
|2026-05-07 08:00:00|2026-05-07 09:00:00|2522.67    |
+-------------------+-------------------+-----------+
only showing top 1 row



In [15]:
####### Praca domowa #######

# 2.
category_window = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .count()
    .filter(col("window.start").cast("string").contains("09:00:00"))
    .orderBy("category")
)

print("Liczba transakcji per kategoria w oknie 09:00-09:30:")
category_window.select("category", col("count").alias("liczba_tx")).show()

Liczba transakcji per kategoria w oknie 09:00-09:30:
+-----------+---------+
|   category|liczba_tx|
+-----------+---------+
|elektronika|      393|
|    książki|      408|
|     odzież|      406|
|    żywność|      391|
+-----------+---------+



In [16]:
####### Praca domowa #######

# 3.
from pyspark.sql.functions import desc

peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .count()
    .orderBy(desc("count"))
)

print("Szczyt transakcji (najbardziej obciążone 15 minut):")
peak_15min.select(
    col("window.start").alias("start_okna"),
    col("window.end").alias("koniec_okna"),
    col("count").alias("liczba_transakcji")
).show(1, truncate=False)

Szczyt transakcji (najbardziej obciążone 15 minut):
+-------------------+-------------------+-----------------+
|start_okna         |koniec_okna        |liczba_transakcji|
+-------------------+-------------------+-----------------+
|2026-05-07 10:15:00|2026-05-07 10:30:00|877              |
+-------------------+-------------------+-----------------+
only showing top 1 row



In [17]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/
	MR140918-PD1.ipynb
	MR140918-PD2.ipynb
	data/
	pd1_robocze.ipynb
	transaction_data_generator.ipynb

nothing added to commit but untracked files present (use "git add" to track)


In [18]:
!git add MR140918-PD2.ipynb

In [19]:
!git commit -m "Lab 2 - homework - Spark batch processing solutions"

[main 9aeeb5b] Lab 2 - homework - Spark batch processing solutions
 1 file changed, 642 insertions(+)
 create mode 100644 MR140918-PD2.ipynb


In [ ]:
!git push origin main

Username for 'https://github.com': 